In [ ]:
# Step 1: Install and import libraries
import os
import sys
import glob
import random
import warnings
import subprocess
import importlib.util
warnings.filterwarnings("ignore")

for package, module in [
    ("kagglehub", "kagglehub"),
    ("imbalanced-learn", "imblearn")
]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
    confusion_matrix, classification_report, roc_curve,
    precision_recall_curve
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.base import clone
from scipy.stats import binomtest
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Step 2: Download and load the dataset
DATASET_SLUG = "ahmedd4/employee-promotion-prediction-dataset"

try:
    dataset_path = kagglehub.dataset_download(DATASET_SLUG)
    csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
    if not csv_files:
        raise FileNotFoundError("No CSV files were found in the downloaded dataset.")
    data_file = max(csv_files, key=os.path.getsize)
    df = pd.read_csv(data_file)
    print("Loaded:", data_file)
except Exception as e:
    print("Automatic download failed:", str(e))
    from google.colab import files
    uploaded = files.upload()
    csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
    if not csv_files:
        raise FileNotFoundError("Upload a CSV file containing the employee dataset.")
    data_file = csv_files[0]
    df = pd.read_csv(data_file)

df.columns = df.columns.astype(str).str.strip()
print("Dataset shape:", df.shape)
display(df.head())

# Step 3: Identify the target column
target_candidates = [
    "is_promoted", "promotion", "promoted", "promotion_status",
    "promotion_label", "promotion_category", "promotion_result",
    "target", "label", "class", "performance_category"
]

normalized_columns = {
    str(c).strip().lower().replace(" ", "_").replace("-", "_"): c
    for c in df.columns
}

TARGET = next(
    (normalized_columns[c] for c in target_candidates if c in normalized_columns),
    None
)

if TARGET is None:
    likely = [
        c for c in df.columns
        if any(k in c.lower() for k in ["promot", "target", "label", "outcome"])
    ]
    print("Possible target columns:", likely)
    raise ValueError(
        "Target column was not identified automatically. Set TARGET to the actual promotion-label column."
    )

df = df.dropna(subset=[TARGET]).drop_duplicates().reset_index(drop=True)
print("Target:", TARGET)
print(df[TARGET].value_counts(dropna=False))

# Step 4: Encode the target and verify binary classification
target_series = df[TARGET].astype(str).str.strip()
target_lower = target_series.str.lower()

positive_names = {
    "1", "yes", "true", "promoted", "promotion", "promote",
    "high", "positive", "approved"
}
negative_names = {
    "0", "no", "false", "not promoted", "non-promoted",
    "not_promoted", "negative", "low", "rejected"
}

unique_targets = list(target_series.unique())

if len(unique_targets) != 2:
    raise ValueError(
        f"Binary CARE-Net code requires exactly 2 target classes; found {unique_targets}."
    )

positive_matches = [
    v for v in unique_targets if v.lower() in positive_names
]
negative_matches = [
    v for v in unique_targets if v.lower() in negative_names
]

if positive_matches and negative_matches:
    positive_label = positive_matches[0]
    negative_label = negative_matches[0]
    y = (target_series == positive_label).astype(np.int64).to_numpy()
else:
    encoder = LabelEncoder()
    encoded = encoder.fit_transform(target_series)
    y = encoded.astype(np.int64)
    positive_label = encoder.classes_[1]
    negative_label = encoder.classes_[0]

print("Negative class:", negative_label)
print("Positive class:", positive_label)
print("Class prevalence:", pd.Series(y).value_counts(normalize=True).sort_index().to_dict())

# Step 5: Prepare predictors and apply domain-specific feature engineering
X = df.drop(columns=[TARGET]).copy()

id_columns = [
    c for c in X.columns
    if c.lower().strip() in {
        "employee_id", "employeeid", "record_id", "index", "unnamed: 0"
    }
]
X = X.drop(columns=id_columns, errors="ignore")

lower_map = {c.lower().strip(): c for c in X.columns}

def find_column(candidates):
    for candidate in candidates:
        if candidate in lower_map:
            return lower_map[candidate]
    return None

projects_col = find_column(["projects_completed", "projects", "completed_projects"])
experience_col = find_column(["experience_years", "years_of_experience", "experience", "years_experience"])
working_col = find_column(["working_hours_per_week", "working_hours", "hours_per_week", "weekly_working_hours"])
overtime_col = find_column(["overtime_hours", "overtime_hours_per_week", "weekly_overtime_hours"])
training_col = find_column(["training_hours", "training_hours_per_year", "training"])
satisfaction_col = find_column(["job_satisfaction", "satisfaction", "job_satisfaction_level"])

if projects_col and experience_col:
    denominator = pd.to_numeric(X[experience_col], errors="coerce").clip(lower=1)
    X["productivity_index"] = pd.to_numeric(
        X[projects_col], errors="coerce"
    ) / denominator

if working_col:
    weekly_work = pd.to_numeric(X[working_col], errors="coerce")
    if overtime_col:
        weekly_work = weekly_work + pd.to_numeric(X[overtime_col], errors="coerce").fillna(0)
    X["workload_ratio"] = weekly_work / 40.0

if training_col and satisfaction_col:
    satisfaction = X[satisfaction_col]
    if not pd.api.types.is_numeric_dtype(satisfaction):
        satisfaction = satisfaction.astype(str).str.strip().str.lower().map({
            "low": 1, "medium": 2, "high": 3,
            "poor": 1, "moderate": 2, "good": 3,
            "very low": 1, "neutral": 2, "very high": 3
        })
    satisfaction = pd.to_numeric(satisfaction, errors="coerce")
    X["engagement_score"] = (
        pd.to_numeric(X[training_col], errors="coerce") * satisfaction
    )

X = X.replace([np.inf, -np.inf], np.nan)

for col in X.columns:
    if X[col].dtype == "object":
        X[col] = X[col].astype("string").fillna("Missing").astype(str)

numeric_columns = X.select_dtypes(include=np.number).columns.tolist()
categorical_columns = [c for c in X.columns if c not in numeric_columns]

print("Numeric features:", len(numeric_columns))
print("Categorical features:", len(categorical_columns))
print("Engineered features:", [
    c for c in ["productivity_index", "workload_ratio", "engagement_score"]
    if c in X.columns
])

# Step 6: Create stratified train, validation, and test partitions
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_columns),
    ("cat", categorical_pipeline, categorical_columns)
], remainder="drop", verbose_feature_names_out=False)

try:
    X_train = preprocessor.fit_transform(X_train_raw)
    X_val = preprocessor.transform(X_val_raw)
    X_test = preprocessor.transform(X_test_raw)
except TypeError:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False))
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_columns),
        ("cat", categorical_pipeline, categorical_columns)
    ], remainder="drop", verbose_feature_names_out=False)
    X_train = preprocessor.fit_transform(X_train_raw)
    X_val = preprocessor.transform(X_val_raw)
    X_test = preprocessor.transform(X_test_raw)

X_train = np.asarray(X_train, dtype=np.float32)
X_val = np.asarray(X_val, dtype=np.float32)
X_test = np.asarray(X_test, dtype=np.float32)

input_dim = X_train.shape[1]
print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)

# Step 7: Define CARE-Net
class CARENet(nn.Module):
    def __init__(
        self,
        input_dim,
        groups=3,
        d_model=128,
        heads=4,
        group_layers=3,
        fusion_layers=2,
        dropout=0.30,
        use_grouping=True,
        use_inter_attention=True,
        use_dual_path=True,
        use_engineered=True,
        use_prs=True
    ):
        super().__init__()
        self.input_dim = input_dim
        self.groups = min(groups, input_dim)
        self.use_grouping = use_grouping
        self.use_inter_attention = use_inter_attention
        self.use_dual_path = use_dual_path
        self.use_engineered = use_engineered
        self.use_prs = use_prs

        self.register_buffer(
            "group_ids",
            torch.tensor(
                np.array_split(np.arange(input_dim), self.groups)[0].tolist(),
                dtype=torch.long
            ),
            persistent=False
        )
        self.group_indices = [
            torch.tensor(idx, dtype=torch.long)
            for idx in np.array_split(np.arange(input_dim), self.groups)
        ]

        self.group_encoders = nn.ModuleList()
        for idx in self.group_indices:
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=heads,
                dim_feedforward=256,
                dropout=dropout,
                batch_first=True,
                norm_first=False
            )
            self.group_encoders.append(
                nn.Sequential(
                    nn.Linear(len(idx), d_model),
                    nn.GELU(),
                    nn.LayerNorm(d_model),
                    nn.TransformerEncoder(encoder_layer, num_layers=group_layers)
                )
            )

        self.group_norm = nn.LayerNorm(d_model)

        fusion_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=heads,
            dim_feedforward=256,
            dropout=dropout,
            batch_first=True
        )
        self.inter_group_attention = nn.TransformerEncoder(
            fusion_layer, num_layers=fusion_layers
        )

        self.local_branch = nn.Sequential(
            nn.Linear(d_model * self.groups, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.global_branch = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.GELU()
        )

        self.single_branch = nn.Sequential(
            nn.Linear(d_model * self.groups, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU()
        )

        combined_dim = 128 if self.use_dual_path else 64
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

        self.register_buffer(
            "engineered_mask",
            torch.ones(input_dim),
            persistent=False
        )

    def forward(self, x):
        if not self.use_engineered:
            x = x * self.engineered_mask

        representations = []
        for idx, encoder in zip(self.group_indices, self.group_encoders):
            idx = idx.to(x.device)
            group_x = x.index_select(1, idx)
            representations.append(encoder(group_x.unsqueeze(1)).squeeze(1))

        group_tokens = torch.stack(representations, dim=1)
        group_tokens = self.group_norm(group_tokens)

        if self.use_inter_attention:
            fused_tokens = self.inter_group_attention(group_tokens)
        else:
            fused_tokens = group_tokens

        flattened = fused_tokens.flatten(start_dim=1)

        if self.use_dual_path:
            local = self.local_branch(flattened)
            global_features = self.global_branch(fused_tokens.mean(dim=1))
            features = torch.cat([local, global_features], dim=1)
        else:
            features = self.single_branch(flattened)

        logits = self.classifier(features).squeeze(1)

        if not self.use_prs:
            logits = logits

        return logits

# Step 8: Define training and prediction functions
class FocalBCEWithLogits(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.register_buffer("pos_weight", pos_weight)

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, reduction="none", pos_weight=self.pos_weight
        )
        probabilities = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probabilities, 1 - probabilities)
        return (((1 - pt) ** self.gamma) * bce).mean()

class WeightedBCEWithLogits(nn.Module):
    def __init__(self, pos_weight=None):
        super().__init__()
        self.register_buffer("pos_weight", pos_weight)

    def forward(self, logits, targets):
        return F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight
        )

def make_loader(X, y, batch_size=64, shuffle=False):
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32)
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

def train_neural_model(
    model, Xtr, ytr, Xva, yva,
    epochs=60, batch_size=64, lr=1e-3,
    patience=8, focal=True
):
    model = model.to(DEVICE)

    classes = np.unique(ytr.astype(int))
    weights = compute_class_weight(
        class_weight="balanced", classes=classes, y=ytr.astype(int)
    )
    class_weight_map = dict(zip(classes, weights))
    positive_weight = torch.tensor(
        class_weight_map[1] / class_weight_map[0],
        dtype=torch.float32,
        device=DEVICE
    )

    if focal:
        criterion = FocalBCEWithLogits(gamma=2.0, pos_weight=positive_weight)
    else:
        criterion = WeightedBCEWithLogits(pos_weight=positive_weight)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=0.01
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-6
    )

    train_loader = make_loader(Xtr, ytr, batch_size, True)
    val_loader = make_loader(Xva, yva, batch_size, False)

    best_loss = float("inf")
    best_state = None
    wait = 0
    history = {
        "train_loss": [], "val_loss": [],
        "train_accuracy": [], "val_accuracy": []
    }

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        correct = 0
        count = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item() * len(yb)
            correct += ((torch.sigmoid(logits) >= 0.5) == (yb >= 0.5)).sum().item()
            count += len(yb)

        train_loss = total_loss / max(count, 1)
        train_accuracy = correct / max(count, 1)

        model.eval()
        val_total = 0.0
        val_correct = 0
        val_count = 0

        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_total += loss.item() * len(yb)
                val_correct += (
                    (torch.sigmoid(logits) >= 0.5) == (yb >= 0.5)
                ).sum().item()
                val_count += len(yb)

        val_loss = val_total / max(val_count, 1)
        val_accuracy = val_correct / max(val_count, 1)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_accuracy"].append(val_accuracy)

        scheduler.step()

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            wait = 0
        else:
            wait += 1

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1:03d}/{epochs} | "
                f"Train loss {train_loss:.4f} | "
                f"Val loss {val_loss:.4f} | "
                f"Val accuracy {val_accuracy:.4f}"
            )

        if wait >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history

def predict_probabilities(model, X, batch_size=256):
    model.eval()
    probabilities = []
    loader = DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32)),
        batch_size=batch_size,
        shuffle=False
    )
    with torch.no_grad():
        for (xb,) in loader:
            logits = model(xb.to(DEVICE))
            probabilities.extend(torch.sigmoid(logits).cpu().numpy())
    return np.asarray(probabilities)

def calculate_metrics(y_true, probabilities, threshold=0.5):
    predictions = (probabilities >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1": f1_score(y_true, predictions, zero_division=0),
        "MCC": matthews_corrcoef(y_true, predictions),
        "ROC-AUC": roc_auc_score(y_true, probabilities),
        "PR-AUC": average_precision_score(y_true, probabilities)
    }

# Step 9: Train CARE-Net
model = CARENet(
    input_dim=input_dim,
    groups=3,
    d_model=128,
    heads=4,
    group_layers=3,
    fusion_layers=2,
    dropout=0.30
)

model, history = train_neural_model(
    model, X_train, y_train, X_val, y_val,
    epochs=60, batch_size=64, lr=1e-3, patience=8, focal=True
)

val_prob = predict_probabilities(model, X_val)
test_prob = predict_probabilities(model, X_test)
test_pred = (test_prob >= 0.5).astype(int)

# Step 10: Evaluate CARE-Net on the held-out test set
care_metrics = calculate_metrics(y_test, test_prob)
print("CARE-Net test metrics")
display(pd.DataFrame([care_metrics]).round(4))

print(classification_report(
    y_test, test_pred,
    target_names=["Non-promoted", "Promoted"],
    digits=4,
    zero_division=0
))

cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
print("CARE-Net confusion matrix")
display(pd.DataFrame(
    cm,
    index=["Actual non-promoted", "Actual promoted"],
    columns=["Predicted non-promoted", "Predicted promoted"]
))

# Step 11: Plot training and validation curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(history["train_accuracy"], label="Training accuracy")
axes[0].plot(history["val_accuracy"], label="Validation accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("CARE-Net Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(history["train_loss"], label="Training loss")
axes[1].plot(history["val_loss"], label="Validation loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("CARE-Net Loss")
axes[1].legend()
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Step 12: Plot the CARE-Net confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Non-promoted", "Promoted"],
    yticklabels=["Non-promoted", "Promoted"]
)
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.title("CARE-Net Confusion Matrix")
plt.tight_layout()
plt.show()

# Step 13: Train baseline machine-learning models
baseline_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=SEED
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=None,
        class_weight="balanced", random_state=SEED, n_jobs=-1
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300, class_weight="balanced",
        random_state=SEED, n_jobs=-1
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.08, random_state=SEED
    )
}

results = {"CARE-Net": care_metrics}
baseline_probabilities = {"CARE-Net": test_prob}
baseline_predictions = {"CARE-Net": test_pred}

for name, baseline in baseline_models.items():
    print("Training:", name)
    baseline.fit(X_train, y_train)
    val_p = baseline.predict_proba(X_val)[:, 1] if hasattr(baseline, "predict_proba") else baseline.decision_function(X_val)
    test_p = baseline.predict_proba(X_test)[:, 1] if hasattr(baseline, "predict_proba") else baseline.decision_function(X_test)

    if np.min(test_p) < 0 or np.max(test_p) > 1:
        test_p = (test_p - np.min(test_p)) / (np.max(test_p) - np.min(test_p) + 1e-12)
        val_p = (val_p - np.min(val_p)) / (np.max(val_p) - np.min(val_p) + 1e-12)

    pred = (test_p >= 0.5).astype(int)
    results[name] = calculate_metrics(y_test, test_p)
    baseline_probabilities[name] = test_p
    baseline_predictions[name] = pred

# Step 14: Compare all models
results_df = pd.DataFrame(results).T
display((results_df * 100).round(2))

# Step 15: Plot ROC curves for all models
plt.figure(figsize=(8, 7))
for name, probabilities in baseline_probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    auc_value = roc_auc_score(y_test, probabilities)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC={auc_value:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves on the Held-out Test Set")
plt.legend(loc="lower right", fontsize=8)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Step 16: Plot precision-recall curves
plt.figure(figsize=(8, 7))
for name, probabilities in baseline_probabilities.items():
    precision_curve, recall_curve, _ = precision_recall_curve(y_test, probabilities)
    ap = average_precision_score(y_test, probabilities)
    plt.plot(recall_curve, precision_curve, label=f"{name} (PR-AUC={ap:.3f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves")
plt.legend(loc="best", fontsize=8)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Step 17: Run component ablation experiments
ablation_settings = {
    "CARE-Net": {},
    "Without contextual grouping": {"use_grouping": False},
    "Without inter-group attention": {"use_inter_attention": False},
    "Without dual-path architecture": {"use_dual_path": False},
    "Without engineered features": {"use_engineered": False},
    "Without PRS component": {"use_prs": False},
    "BCE loss only": {"loss_type": "bce"}
}

ablation_results = []

for ablation_name, settings in ablation_settings.items():
    if ablation_name == "CARE-Net":
        probabilities = test_prob
        metrics = calculate_metrics(y_test, probabilities)
        ablation_results.append({"Configuration": ablation_name, **metrics})
        continue

    loss_type = settings.pop("loss_type", "focal")
    model_args = {
        "input_dim": input_dim,
        "groups": 3,
        "d_model": 128,
        "heads": 4,
        "group_layers": 3,
        "fusion_layers": 2,
        "dropout": 0.30
    }
    model_args.update(settings)

    ablation_model = CARENet(**model_args)
    ablation_model, _ = train_neural_model(
        ablation_model, X_train, y_train, X_val, y_val,
        epochs=60, batch_size=64, lr=1e-3,
        patience=8, focal=(loss_type == "focal")
    )
    probabilities = predict_probabilities(ablation_model, X_test)
    metrics = calculate_metrics(y_test, probabilities)
    ablation_results.append({"Configuration": ablation_name, **metrics})

ablation_df = pd.DataFrame(ablation_results)
display(ablation_df.round(4))

# Step 18: Perform stratified five-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []

for fold, (fit_idx, valid_idx) in enumerate(cv.split(X_train, y_train), start=1):
    fold_model = CARENet(
        input_dim=input_dim, groups=3, d_model=128,
        heads=4, group_layers=3, fusion_layers=2, dropout=0.30
    )
    fold_model, _ = train_neural_model(
        fold_model,
        X_train[fit_idx], y_train[fit_idx],
        X_train[valid_idx], y_train[valid_idx],
        epochs=60, batch_size=64, lr=1e-3, patience=8
    )
    fold_prob = predict_probabilities(fold_model, X_train[valid_idx])
    fold_metrics = calculate_metrics(y_train[valid_idx], fold_prob)
    cv_rows.append({"Fold": fold, **fold_metrics})
    print("Completed fold:", fold)

cv_df = pd.DataFrame(cv_rows)
display(cv_df.round(4))

cv_summary = cv_df.drop(columns="Fold").agg(["mean", "std"]).T
display(cv_summary.round(4))

# Step 19: Compare paired test predictions using McNemar's test
def mcnemar_exact(y_true, pred_a, pred_b):
    a_correct = pred_a == y_true
    b_correct = pred_b == y_true
    n01 = int(np.sum(a_correct & ~b_correct))
    n10 = int(np.sum(~a_correct & b_correct))
    discordant = n01 + n10
    p_value = 1.0 if discordant == 0 else binomtest(
        min(n01, n10), discordant, 0.5, alternative="two-sided"
    ).pvalue
    return n01, n10, p_value

comparison_rows = []
for name, pred in baseline_predictions.items():
    if name == "CARE-Net":
        continue
    n01, n10, p_value = mcnemar_exact(y_test, test_pred, pred)
    comparison_rows.append({
        "Baseline": name,
        "CARE-Net accuracy (%)": 100 * accuracy_score(y_test, test_pred),
        "Baseline accuracy (%)": 100 * accuracy_score(y_test, pred),
        "Difference (percentage points)": 100 * (
            accuracy_score(y_test, test_pred) - accuracy_score(y_test, pred)
        ),
        "CARE-Net correct / baseline wrong": n01,
        "CARE-Net wrong / baseline correct": n10,
        "McNemar p": p_value
    })

comparison_df = pd.DataFrame(comparison_rows)

def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    n = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.empty(n, dtype=float)
    running_max = 0.0
    for rank, idx in enumerate(order):
        value = min(1.0, (n - rank) * p_values[idx])
        running_max = max(running_max, value)
        adjusted[idx] = running_max
    return adjusted

if not comparison_df.empty:
    comparison_df["Holm-adjusted p"] = holm_adjust(comparison_df["McNemar p"].values)
    display(comparison_df.round(5))

# Step 20: Evaluate threshold sensitivity
threshold_rows = []
for threshold in np.arange(0.30, 0.81, 0.05):
    metrics = calculate_metrics(y_test, test_prob, threshold=float(threshold))
    metrics["Threshold"] = round(float(threshold), 2)
    metrics["Selection rate (%)"] = 100 * np.mean(test_prob >= threshold)
    threshold_rows.append(metrics)

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df.round(4))

plt.figure(figsize=(8, 5))
plt.plot(
    threshold_df["Threshold"], threshold_df["Selection rate (%)"],
    marker="o"
)
plt.xlabel("Decision threshold")
plt.ylabel("Positive prediction rate (%)")
plt.title("CARE-Net Threshold Sensitivity")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Step 21: Estimate calibration metrics and plot a reliability curve
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, log_loss

brier = brier_score_loss(y_test, test_prob)
logloss = log_loss(y_test, np.clip(test_prob, 1e-7, 1 - 1e-7))

fraction_positive, mean_predicted = calibration_curve(
    y_test, test_prob, n_bins=10, strategy="uniform"
)

print("Brier score:", round(brier, 5))
print("Log loss:", round(logloss, 5))

plt.figure(figsize=(6.5, 6))
plt.plot(mean_predicted, fraction_positive, marker="o", label="CARE-Net")
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed positive fraction")
plt.title("CARE-Net Calibration Curve")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Step 22: Plot probability distributions and selective prediction
plt.figure(figsize=(8, 5))
plt.hist(test_prob[y_test == 0], bins=25, alpha=0.6, label="Non-promoted")
plt.hist(test_prob[y_test == 1], bins=25, alpha=0.6, label="Promoted")
plt.xlabel("Predicted promotion probability")
plt.ylabel("Number of records")
plt.title("CARE-Net Predicted Probability Distribution")
plt.legend()
plt.tight_layout()
plt.show()

confidence = np.maximum(test_prob, 1 - test_prob)
selective_rows = []

for coverage in np.arange(0.50, 1.001, 0.05):
    cutoff = np.quantile(confidence, 1 - coverage)
    selected = confidence >= cutoff
    if selected.sum() == 0:
        continue
    selective_rows.append({
        "Coverage (%)": 100 * selected.mean(),
        "Selected records": int(selected.sum()),
        "Selective accuracy (%)": 100 * accuracy_score(
            y_test[selected], test_pred[selected]
        )
    })

selective_df = pd.DataFrame(selective_rows).drop_duplicates("Coverage (%)")
display(selective_df.round(2))

plt.figure(figsize=(7, 5))
plt.plot(
    selective_df["Coverage (%)"],
    selective_df["Selective accuracy (%)"],
    marker="o"
)
plt.xlabel("Coverage (%)")
plt.ylabel("Accuracy on selected records (%)")
plt.title("Selective Prediction Performance")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Step 23: Analyze feature importance using permutation importance
from sklearn.inspection import permutation_importance

feature_names = preprocessor.get_feature_names_out()
rf_model = baseline_models["Random Forest"]
perm = permutation_importance(
    rf_model, X_test, y_test,
    scoring="f1", n_repeats=5, random_state=SEED, n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": perm.importances_mean,
    "Std": perm.importances_std
}).sort_values("Importance", ascending=False).head(20)

display(importance_df)

plt.figure(figsize=(9, 7))
sns.barplot(
    data=importance_df,
    x="Importance", y="Feature"
)
plt.title("Top 20 Permutation Importances of Random Forest")
plt.tight_layout()
plt.show()

# Step 24: Export result tables and predictions
os.makedirs("CARE_Net_Results", exist_ok=True)

results_df.to_csv("CARE_Net_Results/model_comparison.csv")
ablation_df.to_csv("CARE_Net_Results/ablation_results.csv", index=False)
cv_df.to_csv("CARE_Net_Results/five_fold_results.csv", index=False)
cv_summary.to_csv("CARE_Net_Results/five_fold_summary.csv")
threshold_df.to_csv("CARE_Net_Results/threshold_sensitivity.csv", index=False)
selective_df.to_csv("CARE_Net_Results/selective_prediction.csv", index=False)
importance_df.to_csv("CARE_Net_Results/feature_importance.csv", index=False)

if not comparison_df.empty:
    comparison_df.to_csv("CARE_Net_Results/statistical_comparison.csv", index=False)

pd.DataFrame({
    "Actual": y_test,
    "CARE_Net_Probability": test_prob,
    "CARE_Net_Prediction": test_pred
}).to_csv("CARE_Net_Results/test_predictions.csv", index=False)

print("Implementation completed.")
print("Results saved in CARE_Net_Results/")
print("Held-out test size:", len(y_test))
print("CARE-Net metrics:", {k: round(v, 4) for k, v in care_metrics.items()})